# AB-RAG on Colab: full traces, the improvements I1 / I2 / R1, and the report
Answers each of the 500 test questions at every round (K = 5, 10, 15) with the local model, with the paper's retrieval and, on HotpotQA, again with **R1** (the paper's candidate pool reranked by bge-reranker-base, picked on dev questions; on TriviaQA no reranker beat the paper's on dev, so R1 is not run there). Then **I1** shortens every round's answer (`13_answer_extract.py --traces`, the frozen prompt), `10_policies.py` replays static RAG, AB-RAG, AB-RAG + **I2** (most confident round), the budget-matched random control and the oracle on each traces file, and `12_report.py` rebuilds the report with paired tests against the base pipeline. N1 is set aside and not run.

**Once:** on your machine run `python notebooks/make_code_zip.py` and upload `abrag_code.zip` to `MyDrive/abrag/` (upload it again after any code change).

**Each session:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**, and keep this tab open (Colab stops sessions whose tab is closed or idle). Every answer is written to Drive as soon as it exists, so after a disconnect just **Run all** again: answered questions are skipped.

Results and the report end up in `MyDrive/abrag/results/`; the comparison is in `report/REPORT.md`, section 10.

In [ ]:
import time
SESSION_START = time.time()

DATASETS = ["hotpotqa", "triviaqa"]
SPLIT = "test"                # "test" = the 500 test questions; "all" adds the 300 dev questions
RETRIEVERS = {"hotpotqa": ["paper", "r1"],   # one traces file each; R1 = the config's dev pick
              "triviaqa": ["paper"]}         # (r1_retrieval), a no-op on TriviaQA
MODEL = None        # None = the config's Qwen/Qwen2.5-1.5B-Instruct; e.g. "Qwen/Qwen2.5-3B-Instruct"
N_SAMPLES = 5       # sampled answers per round
MAX_HOURS = 11      # stop answering after this many hours from session start

In [ ]:
import glob, json, os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/abrag"
OUT = f"{DRIVE}/results"                      # everything this notebook produces lives on Drive
WORK = "/content/abrag"
assert os.path.exists(f"{DRIVE}/abrag_code.zip"), "upload abrag_code.zip to MyDrive/abrag/ first"
shutil.rmtree(WORK, ignore_errors=True)
shutil.unpack_archive(f"{DRIVE}/abrag_code.zip", WORK)
os.makedirs(OUT, exist_ok=True)
for p in glob.glob(f"{WORK}/results/*"):       # inputs computed on your machine
    dst = f"{OUT}/{os.path.basename(p)}"
    if p.endswith("extract_cache.jsonl") and os.path.exists(dst):   # I1 cache: add the zip's entries
        have = {json.loads(l)["key"] for l in open(dst, encoding="utf-8")}
        with open(dst, "a", encoding="utf-8") as f:
            f.writelines(l for l in open(p, encoding="utf-8") if json.loads(l)["key"] not in have)
    elif not os.path.exists(dst) or os.path.basename(p).startswith(("retrieval_upgrade_", "open_retrieval_")):
        shutil.copy(p, dst)                    # rankings once; retrieval results always from the zip
shutil.rmtree(f"{WORK}/results")
os.symlink(OUT, f"{WORK}/results")
os.chdir(WORK)
sys.path.insert(0, WORK)

def n_lines(path):
    return sum(1 for _ in open(path, encoding="utf-8")) if os.path.exists(path) else 0

def complete(line):
    try:
        json.loads(line)
        return True
    except ValueError:
        return False

for p in glob.glob(f"{OUT}/traces_*.jsonl"):   # a session killed mid-write can leave a cut-off last line
    lines = open(p, encoding="utf-8").read().splitlines()
    good = [l for l in lines if complete(l)]
    if len(good) < len(lines):
        open(p, "w", encoding="utf-8").write("".join(l + "\n" for l in good))
        print(f"{os.path.basename(p)}: dropped {len(lines) - len(good)} cut-off line(s)")

def progress():
    return {os.path.basename(p): n_lines(p) for p in sorted(glob.glob(f"{OUT}/traces_*.jsonl"))
            if not p.endswith("-i1.jsonl")}

print(progress() or "no traces yet")

In [ ]:
def ensure(module, package):
    try:
        __import__(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

for module, package in [("rank_bm25", "rank-bm25"), ("faiss", "faiss-cpu"),
                        ("sentence_transformers", "sentence-transformers")]:
    ensure(module, package)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

In [ ]:
# download the models once up front: fails fast without internet
from huggingface_hub import snapshot_download
from configs.config import CONFIG
models = [CONFIG["embed_model"], MODEL or CONFIG["local_model"], CONFIG["local_model"]]   # the last is I1's extractor
models += [CONFIG["r1_retrieval"][ds][1] for ds in DATASETS if "r1" in RETRIEVERS[ds]]
for m in dict.fromkeys(models):
    snapshot_download(m, allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors"])
    print("ready:", m)

In [ ]:
# one run at a time: two copies of the model ran out of the T4's 15 GB on the longest prompts
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # less memory fragmentation

def traces_cmd(dataset, retriever, hours):
    cmd = [sys.executable, "experiments/09_full_traces.py", "--dataset", dataset, "--split", SPLIT,
           "--backend", "local", "--n_samples", str(N_SAMPLES), "--max_hours", f"{hours:.2f}",
           "--retriever", retriever]
    return cmd + (["--model", MODEL] if MODEL else [])

def hours_left():
    return MAX_HOURS - (time.time() - SESSION_START) / 3600

def log_tail(dataset, chars=1500):   # end of a log (progress bars, errors)
    log = open(f"{OUT}/log_{dataset}.txt", encoding="utf-8", errors="replace").read().replace("\r", "\n")
    return "\n".join(l for l in log.splitlines() if l.strip())[-chars:]

def run_traces(dataset, retriever):
    with open(f"{OUT}/log_{dataset}.txt", "a") as log:
        p = subprocess.Popen(traces_cmd(dataset, retriever, hours_left()), stdout=log, stderr=subprocess.STDOUT)
        last = time.time()
        while p.poll() is None:
            time.sleep(30)
            if time.time() - last > 600:
                print(f"{(time.time() - SESSION_START) / 3600:.1f} h", progress(), flush=True)
                last = time.time()
    return p.returncode

# the paper's retrieval on both datasets first, then R1 (its first run also ranks, a few minutes)
for retriever in ("paper", "r1"):
    for dataset in (ds for ds in DATASETS if retriever in RETRIEVERS[ds]):
        for attempt in range(3):   # a crash loses nothing: the retry resumes at the next unanswered question
            if hours_left() < 0.1:
                break
            print(f"{dataset} / {retriever}: running", progress(), flush=True)
            code = run_traces(dataset, retriever)
            if code == 0:
                break
            print(f"--- {dataset} / {retriever} stopped with an error (exit {code}):\n{log_tail(dataset)}", flush=True)
        else:
            raise RuntimeError(f"{dataset} / {retriever} failed 3 times; send the error above")
print(progress())

In [ ]:
# I1 on every traces file (GPU, minutes), then offline replays and the report (CPU); N1 is set aside
def run(*args):
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(r.stdout[-6000:] + (r.stderr[-2000:] if r.returncode else ""))

base = sorted(p for p in glob.glob(f"{OUT}/traces_*.jsonl") if not p.endswith("-i1.jsonl"))
if base:
    run("experiments/13_answer_extract.py", "--traces", *base)   # writes traces_*-i1.jsonl next to each file
run("experiments/13_answer_extract.py")                          # I1 + I2 on the paper's own answers (logs/)
for p in sorted(glob.glob(f"{OUT}/traces_*.jsonl")):
    dataset, label = os.path.basename(p)[len("traces_"):-len(".jsonl")].split("_", 1)
    print(f"\n===== {dataset} / {label}: {n_lines(p)} questions")
    for split in (("test",) if SPLIT == "test" else ("test", "dev")):
        run("experiments/10_policies.py", "--dataset", dataset, "--backend", label, "--split", split)
run("experiments/12_report.py")
print("report:", f"{OUT}/report/REPORT.md")

**If the session ended early:** Run all again; it resumes. Runs go one at a time: HotpotQA, TriviaQA, then HotpotQA with R1. A traces file is complete at 500 questions (800 with `SPLIT = "all"`); the counts print every 10 minutes and after the run cell. The replay cell works on partial traces (at least 50 test questions), so the report fills in as the runs grow.

**What to read:** `REPORT.md` section 10, "On the new test split": static K=10, AB-RAG, AB-RAG + I2, the random control and the oracle for each pipeline (paper or R1 retrieval, with or without I1), then each pipeline paired question by question against the base AB-RAG, on all 500 questions and on questions 201–500 alone (never used by the paper or while designing I1).

**To read the report locally:** copy `MyDrive/abrag/results/` into the repo's `results/` folder and open `results/report/REPORT.md`, or re-run `python experiments/12_report.py` there.